# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Phạm Văn Hoàng Anh Tú — MSSV 2A202602507** · repo: `tubepvhat1604-bot/K4-DAY02-PhamVanHoangAnhTu-2A202602507`

Notebook chạy trên Google Colab, GPU T4. Code nằm trong `code/*.py`; notebook chỉ gọi hàm.
Checkpoint và log của mọi lần chạy lưu ở Drive `/content/drive/MyDrive/deepweeds_runs/<exp_id>/seed<k>/` (ngoài repo, có resume).
Ảnh giải nén vào đĩa cục bộ `/content/data` (không giải nén vào Drive).

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Ô 0.1 — mount Drive, đặt đường dẫn, import (chạy đầu tiên ở MỌI phiên)
from google.colab import drive
drive.mount('/content/drive')

import os, sys, importlib, platform
REPO_DIR = "/content/drive/MyDrive/K4-DAY02-PhamVanHoangAnhTu-2A202602507"
SUB_DIR  = f"{REPO_DIR}/submissions/2A202602507_pham_van_hoang_anh_tu"
CODE_DIR = f"{SUB_DIR}/code"
RUNS_DIR = "/content/drive/MyDrive/deepweeds_runs"     # checkpoint + log, NGOÀI repo
FIG_DIR  = f"{SUB_DIR}/figures"                         # ảnh EDA/kiểm tra cho báo cáo
os.chdir(SUB_DIR)
for p in (REPO_DIR, CODE_DIR):        # REPO_DIR để import eval.py, CODE_DIR để import code của mình
    if p not in sys.path:
        sys.path.insert(0, p)
os.makedirs(RUNS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

!pip -q install timm openpyxl

import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
print("cwd:", os.getcwd())

In [ ]:
# Ô 0.2 — nạp (lại) code từ các file .py. Sửa file .py xong thì chạy lại ô này, KHÔNG cần khởi động lại phiên.
# (Không dùng %load_ext autoreload: lỗi với Python 3.13 trên Colab.)
import dataset, model, losses, train, pipeline_checks, eval as ev
for m in (dataset, model, losses, train, pipeline_checks, ev):
    importlib.reload(m)
from train import Config, run
print("Đã nạp:", [m.__name__ for m in (dataset, model, losses, train, pipeline_checks, ev)])

### Tải dữ liệu vào đĩa cục bộ `/content/data`
Mỗi phiên Colab chạy lại ô này. Lần đầu tải `images.zip` từ Zenodo (~490 MB) rồi **lưu một bản zip** vào
`MyDrive/deepweeds_data/` (chỉ file zip, ngoài repo) để các phiên sau chép từ Drive cho nhanh. Ảnh luôn giải nén ra `/content/data`.
4 file CSV fold 0 tải nguyên bản từ GitHub của tác giả, không sửa.

In [ ]:
# Ô 0.3 — tải ảnh + CSV, kiểm tra MD5, giải nén (chạy lại mỗi phiên; đã có thì bỏ qua)
import hashlib, shutil, zipfile, urllib.request

DATA_DIR   = "/content/data"
LABELS_DIR = f"{DATA_DIR}/labels"
ZIP_LOCAL  = f"{DATA_DIR}/images.zip"
ZIP_DRIVE  = "/content/drive/MyDrive/deepweeds_data/images.zip"
ZIP_URL    = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5    = "b7b30f96d466fba86016aa5a26606e0f"
SAVE_ZIP_TO_DRIVE = True
os.makedirs(LABELS_DIR, exist_ok=True)

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download(url, path):
    with urllib.request.urlopen(url) as r, open(path + ".part", "wb") as f:
        total, done = int(r.headers.get("Content-Length", 0)), 0
        while chunk := r.read(1 << 22):
            f.write(chunk); done += len(chunk)
            if done % (100 << 20) < (1 << 22):
                print(f"  {done >> 20} / {total >> 20} MB", flush=True)
    os.replace(path + ".part", path)

if not os.path.exists(f"{DATA_DIR}/.images_ok"):
    if not os.path.exists(ZIP_LOCAL):
        if os.path.exists(ZIP_DRIVE):
            print("Chép images.zip từ Drive..."); shutil.copy(ZIP_DRIVE, ZIP_LOCAL)
        else:
            print("Tải images.zip từ Zenodo..."); download(ZIP_URL, ZIP_LOCAL)
    got = md5sum(ZIP_LOCAL)
    assert got == ZIP_MD5, f"MD5 sai: {got}. Xoá {ZIP_LOCAL} (và {ZIP_DRIVE} nếu có) rồi chạy lại ô này."
    print("MD5 đúng:", got)
    if SAVE_ZIP_TO_DRIVE and not os.path.exists(ZIP_DRIVE):
        os.makedirs(os.path.dirname(ZIP_DRIVE), exist_ok=True); shutil.copy(ZIP_LOCAL, ZIP_DRIVE)
        print("Đã lưu bản zip vào", ZIP_DRIVE)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(f"{DATA_DIR}/images")
    open(f"{DATA_DIR}/.images_ok", "w").close()

BASE_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{LABELS_DIR}/{name}.csv"):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}.csv", f"{LABELS_DIR}/{name}.csv")

# Tìm thư mục thật sự chứa ảnh .jpg (zip có thể có thư mục con)
IMAGES_DIR = max((r for r, _, fs in os.walk(f"{DATA_DIR}/images")), key=lambda r: len(os.listdir(r)))
print("IMAGES_DIR =", IMAGES_DIR, "|", sum(f.endswith(".jpg") for f in os.listdir(IMAGES_DIR)), "ảnh .jpg")
print("CSV:", sorted(os.listdir(LABELS_DIR)))

In [ ]:
# Ô 0.4 — cấu hình dùng chung cho MỌI thí nghiệm (đặt ở đây để Restart & Run All chạy được từ trên xuống)
BASE = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=RUNS_DIR,
            curves_dir=f"{SUB_DIR}/curves", pred_dir=f"{SUB_DIR}/predictions", num_workers=2)
print(Config(**BASE))

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Ô B0.1 — kiểm tra chia dữ liệu bắt buộc (S1-S4): số ảnh mỗi tập/lớp, giao rỗng, hợp = 17.509, mọi file tồn tại
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_info = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

In [ ]:
# Ô B0.2 — EDA: phân bố lớp (biểu đồ), đối chiếu Table 1, ảnh mẫu 3 ảnh/lớp, thống kê kích thước ảnh
eda_counts = pipeline_checks.eda(train_df, val_df, test_df, IMAGES_DIR, FIG_DIR, labels_csv=f"{LABELS_DIR}/labels.csv")

In [ ]:
# Ô B0.3 — test tự viết (focal γ=0 ≡ CE, label smoothing, CutMix lam theo diện tích thật, nhóm tham số, EMA...)
#          và test của repo (eval.py + khung starter/)
!cd {CODE_DIR} && python -m unittest test_code 2>&1 | tail -n 3
!cd {REPO_DIR} && python -m unittest discover -s tests 2>&1 | tail -n 3

In [ ]:
# Ô B0.4 — kiểm tra pipeline (slide trang 59, GUIDE mục 1.3)
train.set_seed(0)                                                   # 1) cố định seed
val_loader = dataset.make_loader(val_df, IMAGES_DIR, dataset.build_transforms(False, 224), 64, train=False)
pipeline_checks.initial_loss("resnet50", val_loader)                # 2) loss ban đầu ≈ ln 9 = 2,197
losses_ob = pipeline_checks.overfit_one_batch("resnet50", train_df, IMAGES_DIR, n=16, steps=60)  # 3) overfit 1 batch

In [ ]:
# Ô B0.5 — xem ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn, và ảnh sau CutMix
for aug in ("basic", "trivial"):
    pipeline_checks.show_augmented(train_df, IMAGES_DIR, aug, f"{FIG_DIR}/aug_{aug}.png", n=8)

In [ ]:
# Ô B0.6 — ước lượng thời gian 1 epoch mỗi backbone (30 bước train AMP + suy luận val), để lập ngân sách GPU
BACKBONES = ["resnet50", "convnext_tiny", "deit_small_patch16_224", "efficientnet_b0",
             "mobilenetv3_large_100", "swin_tiny_patch4_window7_224"]
timing = pipeline_checks.time_backbones(BACKBONES, train_df, IMAGES_DIR)
timing.to_csv(f"{FIG_DIR}/timing_estimate.csv", index=False)
timing

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id="B01", backbone=..., seed=0))
#       Gợi ý: from train import Config, run
# TODO: ghi vào bảng: tag trọng số, #params, GMAC, macro-F1 val, top-1 val, thời gian train/epoch.
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py